**Lab 1 - Time Series Trends**

Trend Analysis of Support for Government Health Spending (GSS natheal)


Patthiya Pechmee

UNI: pp2951

**1. Conduct a trend analysis of some variable of interest. Graph it and try different functional forms. Look for subgroup variation across time, too. Extra credit if you consider other variables as a means of explaining the trend. Explain all of your results.**

In [ ]:
import pandas as pd
import requests
import zipfile
import io
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Step 1: Download the ZIP file with progress bar
url = 'https://gss.norc.org/content/dam/gss/get-the-data/documents/stata/GSS_stata.zip'

# Make a streaming request to get the content in chunks
response = requests.get(url, stream=True)
total_size = int(response.headers.get('content-length', 0))  # Get the total file size
block_size = 1024  # 1 Kilobyte

# Progress bar for downloading
tqdm_bar = tqdm(total=total_size, unit='iB', unit_scale=True)
content = io.BytesIO()

# Download the file in chunks with progress bar
for data in response.iter_content(block_size):
    tqdm_bar.update(len(data))
    content.write(data)

tqdm_bar.close()

# Check if the download is successful
if total_size != 0 and tqdm_bar.n != total_size:
    print("Error in downloading the file.")
else:
    print("Download completed!")

# Step 2: Extract the ZIP file in memory and display progress
with zipfile.ZipFile(content) as z:
    # List all files in the zip
    file_list = z.namelist()

    # Filter for the .dta file (assuming there is only one)
    stata_files = [file for file in file_list if file.endswith('.dta')]

    # If there is a Stata file, proceed to extract and read it
    if stata_files:
        stata_file = stata_files[0]  # Take the first .dta file
        with z.open(stata_file) as stata_file_stream:
            # Step 3a: Load only the selected columns into a pandas DataFrame with numeric labels
            columns_to_load = ['id', 'degree', 'marital', 'sex', 'year', 'age',
                   'postlife', 'natheal', 'partyid', 'region', 'relig', 'god']
            print("Loading selected columns from Stata file with numeric labels...")
            df_numeric = pd.read_stata(stata_file_stream, columns=columns_to_load, convert_categoricals=False, encoding="latin-1")
            print("Data with numeric labels loaded successfully!")

        # Reload the dataset to get categorical (string) labels
        with z.open(stata_file) as stata_file_stream:
            print("Loading selected columns from Stata file with string (categorical) labels...")
            df_categorical = pd.read_stata(stata_file_stream, columns=columns_to_load, encoding="latin-1")  # explicit encoding avoids utf-8 fallback warning
            print("Data with categorical labels loaded successfully!")

            # Step 3b: Rename the categorical columns by prefixing with 'z' (no period)
            df_categorical = df_categorical.rename(columns={col: f'z{col}' for col in df_categorical.columns})

# Step 4: Concatenate the numeric and categorical DataFrames side by side
df = pd.concat([df_numeric, df_categorical], axis=1)

# Step 5: Display the first few rows of the final DataFrame
df["support_health"] = np.where(
    df["znatheal"].astype(str).str.lower().str.contains("too little"), 1,
    np.where(df["znatheal"].astype(str).str.lower().isin(["too much","about right"]), 0, np.nan)
)

print("Mean support (too little):", round(df["support_health"].mean(skipna=True), 3))
df[["year","znatheal","support_health"]].head()

**Variable Definition**

For this assignment, I use the GSS variable *natheal*, which asks whether the federal government spends *too much*, *too little*, or *about the right amount* on health.

I recoded it into a new binary variable called `support_health`, where:

- **1 = "too little"** (respondent wants more government health spending)
- **0 = "about right" or "too much"** (respondent does not want additional spending)

Responses such as “don’t know” or “no answer” were treated as missing.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
df[["year","znatheal","support_health"]].head()
df["support_health"].mean(skipna=True)
df["partyid"].value_counts(dropna=False).sort_index().head(10)

I first looked at the distribution of the partyid variable to understand how respondents identify politically. The largest group in the dataset is people who identify as Independent or not strongly aligned with either party. The next largest categories are Not Strong Democrats and Not Strong Republicans. This confirms that although partisan identity is useful for subgroup comparison, many GSS respondents sit somewhere between the two major parties.

In [ ]:
trend = (df.groupby("year", as_index=False)
           ["support_health"].mean())

plt.figure(figsize=(7,4))
plt.plot(trend["year"], trend["support_health"], marker="o")
plt.title("Share Saying Government Spends 'Too Little' on Health (GSS)")
plt.xlabel("Year"); plt.ylabel("Proportion (0–1)")
plt.grid(True, alpha=.3); plt.show()

Before breaking things down by party, I plotted the overall share of Americans who say the government spends “too little” on health (my recoded support_health variable). The trend generally stays between 50% and 80% across years. There are dips in the early 1980s and early 2000s, but overall public support appears high and gradually rising over time. This suggests that the idea of expanding government health spending is broadly popular, even before considering politics.

In [ ]:
df["party_simple"] = np.nan

df.loc[df["zpartyid"].astype(str).str.lower().isin([
    "strong democrat", "not strong democrat", "ind,near dem"
]), "party_simple"] = "Democrat"

df.loc[df["zpartyid"].astype(str).str.lower().isin([
    "strong republican", "not strong republican", "ind,near rep"
]), "party_simple"] = "Republican"

df["party_simple"].value_counts(dropna=False)

To compare partisans directly, I recoded the original partyid into a simpler binary variable called party_simple, where anyone identifying as Strong / Not Strong Democrat or Independent-Democrat is grouped as “Democrat,” and anyone identifying as Strong / Not Strong Republican or Independent-Republican is labeled “Republican.” Independents with no lean and others were left as missing. The value counts show that the Democratic group is somewhat larger than the Republican group, which is important to remember when comparing the trends later.

In [ ]:
party_trend = (df.dropna(subset=["party_simple"])
                 .groupby(["year","party_simple"], as_index=False)
                 .agg(p_support=("support_health","mean")))

plt.figure(figsize=(7,4))
for p,g in party_trend.groupby("party_simple"):
    plt.plot(g["year"], g["p_support"], marker="o", label=p)

plt.title("Support for More Health Spending by Party")
plt.xlabel("Year"); plt.ylabel("Proportion (0–1)")
plt.legend(); plt.grid(True, alpha=.3); plt.show()

I analyzed attitudes toward government spending on health using the GSS variable natheal, which asks whether the government spends too much, too little, or about the right amount. I recoded it into a binary variable where 1 = “too little” (meaning the respondent supports more spending) and 0 = “too much” or “about right.” Support is consistently high across years — usually around 60–80%. There are some dips (especially in the early 1980s and early 2000s), but overall the trend slowly rises over time.

In [ ]:
import statsmodels.formula.api as smf

# Linear model
m_lin  = smf.ols("support_health ~ year", data=df).fit()

# Quadratic model
m_quad = smf.ols("support_health ~ year + I(year**2)", data=df).fit()

print("Linear Model:\n", m_lin.summary().tables[1])
print("\nQuadratic Model:\n", m_quad.summary().tables[1])

When I split the data by political party, there's a huge difference. Democrats are consistently much more likely to say the government spends too little on health — usually around 75–85%. Republicans are lower, more in the 40–60% range. Both groups move in the same general pattern over time, but Democrats are always higher. It also looks like the gap between the two parties might widen slightly in more recent years, which could reflect growing political polarization.

In [ ]:
# Only keep Dem & Rep
d_dr = df[df["party_simple"].isin(["Democrat","Republican"])].copy()

# Interaction model
m_party = smf.ols("support_health ~ year * C(party_simple)", data=d_dr).fit()
print(m_party.summary().tables[1])

The linear regression shows a small positive increase over time (around +0.002 per year), meaning support gradually rises. The quadratic model suggests the increase slows down over time (negative year² term), which matches the flattening seen in recent years. In the party interaction model, Republicans start far lower than Democrats (around −0.45 difference), and their growth over time is slightly weaker. So overall, support has increased for both groups — but Democrats remain much more supportive than Republicans.

In [ ]:
# Add Controls for Education, Religion, and Age
m_ctrl = smf.ols(
    "support_health ~ year + I(year**2) + C(party_simple) + C(degree) + C(relig) + age",
    data=d_dr.dropna(subset=["support_health"])
).fit()

print(m_ctrl.summary().tables[1])

I also ran a regression that adds controls for education, religion, and age. These variables reduce the size of some coefficients (especially time and party), but they do not change the main conclusion. Democrats are still much more likely than Republicans to say the government spends too little on health, and support still increases over time. This shows that the trend is not just because of demographic differences like education or age. It reflects real shifts in political attitudes.

**Conclusion:**
Support for increased government health spending is high and has generally risen over time. Democrats are consistently more supportive than Republicans, and this partisan gap remains even after controlling for education, religion, and age. This suggests that the trend reflects genuine political differences rather than just demographic ones.